# MVP UI Demo: Full Cycle Entry Point

Этот notebook — каноническая пользовательская точка входа поверх полного пайплайна `run_full_cycle`.

- `MODE = "full"` — полное переобучение и пересборка артефактов.
- `MODE = "fast"` — повторный запуск с уже сохраненной моделью.

Базовый сценарий: `Restart Kernel` -> `Run All`.


In [ ]:
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Project root not found (expected pyproject.toml and src/).")

PROJECT_ROOT = find_project_root(Path.cwd())
INPUT_PATH = PROJECT_ROOT / "data" / "raw" / "episodes.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "demo_run"
MODEL_PATH = OUTPUT_DIR / "models" / "hmm_model.pkl"

MODE = "full"  # "full" | "fast"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"INPUT_PATH: {INPUT_PATH}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"MODEL_PATH: {MODEL_PATH}")
print(f"MODE: {MODE}")

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Input Excel not found: {INPUT_PATH}")


In [ ]:
# Optional: ручная очистка перед полным прогоном.
# По умолчанию удаление отключено, чтобы `Run All` был безопасным.

import shutil

RUN_CLEANUP = False  # Переключите в True и запустите ячейку вручную при необходимости

if RUN_CLEANUP:
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)
        print(f"Removed: {OUTPUT_DIR}")
    else:
        print(f"Nothing to remove: {OUTPUT_DIR}")
else:
    print("Cleanup skipped (RUN_CLEANUP=False).")


In [ ]:
from hidden_patterns_combat.app.full_cycle import run_full_cycle

result = run_full_cycle(
    input_path=INPUT_PATH,
    output_dir=OUTPUT_DIR,
    mode=MODE,
    model_path=MODEL_PATH,
)

result.as_dict()


In [ ]:
import pandas as pd
from IPython.display import Markdown, display

diagnostics_csv = OUTPUT_DIR / "diagnostics" / "hmm_state_interpretation.csv"
report_md = OUTPUT_DIR / "reports" / "full_cycle_report.md"

if diagnostics_csv.exists():
    print(f"Diagnostics: {diagnostics_csv}")
    display(pd.read_csv(diagnostics_csv))
else:
    print(f"Diagnostics file not found: {diagnostics_csv}")

if report_md.exists():
    print(f"Report: {report_md}")
    display(Markdown(report_md.read_text(encoding="utf-8")))
else:
    print(f"Report file not found: {report_md}")


In [ ]:
from IPython.display import Image, display

plots_dir = OUTPUT_DIR / "plots"
expected_plots = [
    "hidden_state_sequence.png",
    "state_probability_profile.png",
    "transition_distribution.png",
    "athlete_comparative_profile.png",
    "scenario_success_frequencies.png",
]

shown = set()

if plots_dir.exists():
    for name in expected_plots:
        path = plots_dir / name
        if path.exists():
            print(f"Showing: {path}")
            display(Image(filename=str(path)))
            shown.add(path.name)
        else:
            print(f"Missing plot (skip): {path}")

    for extra_path in sorted(plots_dir.glob("*.png")):
        if extra_path.name not in shown:
            print(f"Showing extra plot: {extra_path}")
            display(Image(filename=str(extra_path)))
else:
    print(f"Plots directory not found: {plots_dir}")


In [ ]:
print("Run completed. Artifacts:")
print(f"- Model: {MODEL_PATH}")
print(f"- Diagnostics: {OUTPUT_DIR / 'diagnostics'}")
print(f"- Plots: {OUTPUT_DIR / 'plots'}")
print(f"- Report: {OUTPUT_DIR / 'reports' / 'full_cycle_report.md'}")
